# yeppoh playground

Run cells top to bottom. You'll see visuals within 30 seconds.

**No Genesis needed** for the first sections — just torch + matplotlib.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 8)
plt.rcParams['figure.facecolor'] = 'black'
print('ready')

---
## 1. Turing Patterns (instant beauty)

This is what creature skin/surface patterns will look like.
Gray-Scott reaction-diffusion — two chemicals interacting on a grid.

In [ ]:
from yeppoh.world.reaction_diffusion import ReactionDiffusion

# Spots pattern
rd = ReactionDiffusion(grid_size=256, f=0.055, k=0.062, device='cpu')
for _ in range(2000):
    rd.step(n_steps=1)

plt.imshow(rd.get_pattern_image().numpy(), cmap='inferno')
plt.axis('off')
plt.title('Turing pattern: spots', color='white', fontsize=16)
plt.show()

In [ ]:
# Try different f/k values — each produces a different pattern
patterns = {
    'spots':     (0.055, 0.062),
    'stripes':   (0.039, 0.058),
    'waves':     (0.026, 0.051),
    'worms':     (0.078, 0.061),
    'maze':      (0.029, 0.057),
    'holes':     (0.039, 0.065),
}

fig, axes = plt.subplots(2, 3, figsize=(15, 10), facecolor='black')
for ax, (name, (f, k)) in zip(axes.flat, patterns.items()):
    rd = ReactionDiffusion(grid_size=256, f=f, k=k, device='cpu')
    for _ in range(3000):
        rd.step(n_steps=1)
    ax.imshow(rd.get_pattern_image().numpy(), cmap='inferno')
    ax.set_title(f'{name}  (f={f}, k={k})', color='white', fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Animate a pattern growing — watch it self-organize
rd = ReactionDiffusion(grid_size=256, f=0.039, k=0.058, device='cpu')

fig, ax = plt.subplots(figsize=(8, 8), facecolor='black')
img = ax.imshow(rd.get_pattern_image().numpy(), cmap='inferno', vmin=0, vmax=0.5)
ax.axis('off')
ax.set_title('growing...', color='white')

def animate(frame):
    rd.step(n_steps=50)  # 50 sub-steps per frame
    img.set_data(rd.get_pattern_image().numpy())
    ax.set_title(f'step {frame * 50}', color='white')
    return [img]

anim = animation.FuncAnimation(fig, animate, frames=80, interval=50, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())

---
## 2. Pheromone Diffusion

This is how creatures communicate through the environment.
Drop chemicals, watch them spread and fade.

In [ ]:
from yeppoh.world.pheromones import PheromoneGrid

grid = PheromoneGrid(n_channels=3, grid_size=64, device='cpu')

# Drop 3 different chemicals at different positions
grid.emit(torch.tensor([[0.0, 0.0, 0.0]]), channel=0, amount=5.0)   # red
grid.emit(torch.tensor([[1.0, 1.0, 0.0]]), channel=1, amount=5.0)   # green
grid.emit(torch.tensor([[-1.0, 0.5, 0.0]]), channel=2, amount=5.0)  # blue

# Simulate diffusion and capture frames
frames = []
for step in range(200):
    grid.step()
    if step % 5 == 0:
        # Take a 2D slice at z=32 (middle of grid)
        z_mid = grid.grid_size // 2
        rgb = torch.stack([
            grid.grid[0, :, :, z_mid],
            grid.grid[1, :, :, z_mid],
            grid.grid[2, :, :, z_mid],
        ], dim=-1)
        frames.append((rgb / rgb.max().clamp(min=0.01)).numpy())

# Animate
fig, ax = plt.subplots(figsize=(8, 8), facecolor='black')
img = ax.imshow(frames[0])
ax.axis('off')
ax.set_title('pheromone diffusion (RGB = 3 chemicals)', color='white')

def animate(i):
    img.set_data(frames[i])
    ax.set_title(f'pheromone diffusion — step {i * 5}', color='white')
    return [img]

anim = animation.FuncAnimation(fig, animate, frames=len(frames), interval=80, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())

---
## 3. The Creature Brain

What does the neural network look like when it fires?
Let's feed it random observations and see the outputs.

In [ ]:
from yeppoh.brain import CreatureBrain

brain = CreatureBrain(
    obs_dim=99,
    action_dim=47,
    feature_dim=128,
    memory_dim=64,
    use_memory=True,
    use_communication=True,
    use_drives=True,
)

print(f'Brain parameters: {sum(p.numel() for p in brain.parameters()):,}')
print(f'Input: {brain.obs_dim} dims (sensory observations)')
print(f'Output: {brain.action_dim} dims (motor + signal + sensing)')
print()

# Feed it random observations and watch the outputs over 100 steps
# (untrained brain — just seeing the signal flow)
actions_over_time = []
drives_over_time = []
messages_over_time = []
hidden = None
drives = None

for t in range(100):
    obs = torch.randn(1, 99) * 0.5  # random sensory input
    out = brain(obs, hidden=hidden, current_drives=drives)
    hidden = out['hidden']
    drives = out['drives']
    actions_over_time.append(out['action_mean'].detach().squeeze().numpy())
    drives_over_time.append(out['drives'].detach().squeeze().numpy())
    if out['outgoing_message'] is not None:
        messages_over_time.append(out['outgoing_message'].detach().squeeze().numpy())

actions = np.array(actions_over_time)
drives_arr = np.array(drives_over_time)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), facecolor='black')

# Motor actions over time (first 10 dims)
ax = axes[0]
ax.set_facecolor('black')
for i in range(10):
    ax.plot(actions[:, i], alpha=0.7, linewidth=1)
ax.set_title('motor actions (10 of 27 dims)', color='white')
ax.set_ylabel('value', color='white')
ax.tick_params(colors='white')

# Internal drives
ax = axes[1]
ax.set_facecolor('black')
drive_names = ['hunger', 'curiosity', 'fear', 'social']
for i, name in enumerate(drive_names):
    ax.plot(drives_arr[:, i], label=name, linewidth=2)
ax.set_title('internal drives', color='white')
ax.legend(loc='upper right')
ax.tick_params(colors='white')

# Communication messages
if messages_over_time:
    msgs = np.array(messages_over_time)
    ax = axes[2]
    ax.set_facecolor('black')
    ax.imshow(msgs.T, aspect='auto', cmap='coolwarm', interpolation='nearest')
    ax.set_title('outgoing message (16 dims over time)', color='white')
    ax.set_xlabel('timestep', color='white')
    ax.set_ylabel('message dim', color='white')
    ax.tick_params(colors='white')

plt.tight_layout()
plt.show()

---
## 4. Design Your Own Pattern

Tweak f and k to discover new patterns. This is the palette
your creatures will wear.

In [ ]:
# ★ PLAY WITH THESE ★
f = 0.04    # feed rate:  higher = more chemical A
k = 0.060   # kill rate:  higher = more chemical B consumed
# sweet spot is roughly f in [0.02, 0.08], k in [0.04, 0.07]

rd = ReactionDiffusion(grid_size=256, f=f, k=k, device='cpu')
for _ in range(4000):
    rd.step(n_steps=1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), facecolor='black')
ax1.imshow(rd.get_pattern_image().numpy(), cmap='inferno')
ax1.set_title(f'V channel (f={f}, k={k})', color='white')
ax1.axis('off')

ax2.imshow(rd.U.squeeze().numpy(), cmap='viridis')
ax2.set_title('U channel', color='white')
ax2.axis('off')
plt.show()

---
## 5. Genesis Physics (requires `pip install genesis-world`)

Skip this section if Genesis isn't installed yet.
This shows actual soft-body creatures in the physics sim.

In [ ]:
# Uncomment and run when Genesis is installed:
#
# import genesis as gs
# gs.init(backend=gs.metal)  # or gs.cpu on non-Apple machines
#
# scene = gs.Scene(
#     sim_options=gs.options.SimOptions(dt=0.01, substeps=2),
#     show_viewer=True,
# )
# scene.add_entity(gs.morphs.Plane())
#
# blob = scene.add_entity(
#     material=gs.materials.MPM.Elastic(E=5000, nu=0.45, rho=1000),
#     morph=gs.morphs.Sphere(pos=(0, 0, 0.5), radius=0.3),
# )
#
# scene.build(n_envs=1)
#
# # Run 500 steps — watch the blob fall and squish
# for _ in range(500):
#     scene.step()
#
# print('done — close the viewer window')

---
## What's Next?

| Ready to... | Do this |
|---|---|
| **See a creature train** | `python experiments/01_basic_blob.py` (terminal) |
| **Watch it live** | Add `scene.show_viewer=true scene.n_envs=1` to the command |
| **Design a new body** | Edit `yeppoh/body/morphology.py` |
| **Design a new pattern** | Copy cell 4 above, tweak f and k |
| **Train on cloud** | `python scripts/modal_train.py --experiment 02_locomotion` |